# 🏗️ Notebook 1: S3 (Object Storage) — Requirements & Architecture

Welcome! In this lab we're going to design a simplified version of
**Amazon S3** — a service that stores *objects* (arbitrary blobs of
bytes) behind a simple HTTP API.

By the end of this notebook you should be able to answer:

- What is object storage, and how is it different from a file system?
- Why is durability such a big deal, and how do we even *measure* it?
- What does the 10,000-foot architecture look like?

The next two notebooks zoom in on the **data model & APIs** and on a
few **deep-dive algorithms** (presigned URLs, consistent hashing,
erasure coding, multipart uploads).


## 🛠️ Setup

```bash
cd 06-system-designs/s3
uv sync
```

Then in VS Code pick the `.venv` kernel from the top-right of the notebook. If
it doesn't show up: `Cmd+Shift+P` → **Reload Window** and try again.

Everything in this lab is **pure Python** — no databases, no Docker. You can
run it on a laptop in a few seconds.


## 🧠 What *is* object storage?

Three ways people store data on a computer:

| Kind | Analogy | Access pattern |
|---|---|---|
| **Block storage** (e.g. EBS, a hard drive) | A raw notebook of empty pages | Read/write specific offsets — the OS decides what goes where |
| **File storage** (e.g. NFS, your laptop's disk) | A filing cabinet with folders and files | Open/read/write/seek, rename, permissions |
| **Object storage** (e.g. S3) | A giant parking lot where each car has a printed ticket | `PUT`/`GET`/`DELETE` a whole blob, addressed by a key |

In object storage, **the smallest unit you can touch is the whole
object**. You can't open `cat.jpg` and patch byte 42 — you replace the
entire thing, and you get a new version. That one rule is what makes
the system scale to exabytes.


## 🎯 Requirements

### Functional
- Create & delete **buckets** (a bucket is just a namespace).
- `PUT`, `GET`, `LIST`, `DELETE` objects inside a bucket.
- **Versioning** per bucket so an accidental overwrite isn't fatal.
- **Multipart upload** for large objects (multi-GB files).
- **Presigned URLs** — short-lived links that grant one specific
  action without handing out your credentials.
- **Cross-region replication** for disaster recovery.

### Non-functional
- **Durability ≈ 11 nines** (99.999999999%). One in 10¹¹ objects
  might be lost in a year. That is the headline number.
- **Availability ≈ 4 nines** (99.99%). A bit of downtime is fine; a
  lost object is not.
- **Scale**: petabytes today, exabytes later. Millions of PUT/s.
- **Cheap**: cost per GB-month matters more than microsecond latency.
- **Consistency**: strong read-after-write for a single key is nice,
  but listing and replication can be eventually consistent.


## ✏️ Back-of-envelope

Before we draw any boxes, let's play with numbers. We'll assume a
modest workload and see where the first bottleneck hits.


In [1]:
# Back-of-envelope estimate
daily_uploads = 100_000_000        # 100 M new objects/day (small workload)
avg_size_bytes = 1 * 1024 * 1024   # 1 MB average object

bytes_per_day = daily_uploads * avg_size_bytes
pb_per_day = bytes_per_day / (1024 ** 5)
print(f"~{pb_per_day:.1f} PB of new data per day")

# Peak write QPS assuming a 6-hour peak window
peak_window_seconds = 6 * 3600
peak_qps = daily_uploads / peak_window_seconds
print(f"~{peak_qps:,.0f} PUT/s at peak")

# 10-year retention
ten_year_eb = pb_per_day * 365 * 10 / 1024
print(f"~{ten_year_eb:.1f} EB over 10 years (ignoring deletes)")


~0.1 PB of new data per day
~4,630 PUT/s at peak
~0.3 EB over 10 years (ignoring deletes)


Takeaway: even this *small* workload produces **exabytes** over a
decade. We can't store that on one box, and we can't afford to keep
3 full copies of it either. Welcome to the real problem.


## 💸 Durability vs cost: replication vs erasure coding

The obvious way to avoid losing data is to keep N copies. The problem
is cost: 3 copies = 3× the hardware bill.

**Erasure coding** is the clever alternative. We split each object
into `k` data shards and compute `m` parity shards. Any `k` of the
`k+m` shards are enough to rebuild the object.

Example: 10+4 means we keep 14 shards total and can lose any 4 of them.


In [2]:
# Storage overhead: replication vs erasure coding
def overhead_replication(copies: int) -> float:
    return copies

def overhead_erasure(k: int, m: int) -> float:
    return (k + m) / k

schemes = [
    ("1x (single copy, no redundancy)", overhead_replication(1)),
    ("3x replication",                  overhead_replication(3)),
    ("Erasure 10+4",                    overhead_erasure(10, 4)),
    ("Erasure 17+3",                    overhead_erasure(17, 3)),
]

raw_pb = 1000  # 1 EB of logical data
print(f"{'scheme':<35} {'overhead':>8}  {'physical PB for 1 EB':>22}")
for name, ov in schemes:
    print(f"{name:<35} {ov:>7.2f}x  {raw_pb*ov:>22,.0f}")


scheme                              overhead    physical PB for 1 EB
1x (single copy, no redundancy)        1.00x                   1,000
3x replication                         3.00x                   3,000
Erasure 10+4                           1.40x                   1,400
Erasure 17+3                           1.18x                   1,176


Going from 3× replication to 10+4 erasure coding cuts physical
storage in **less than half**. At exabyte scale that is billions of
dollars. This is why every serious object store uses erasure coding
for its main tier.

We'll actually implement a tiny erasure code in Notebook 3.


## 🎲 Where does "eleven nines" come from?

Durability is a probability, not a promise. Assume each shard
independently fails in a year with probability `p`. We lose the
object only if *more* than `m` shards fail at the same time — because
any `k` surviving shards can rebuild it.

This little script shows why adding parity shards is so powerful.


In [3]:
# Durability intuition: annual probability of losing one object
from math import comb, log10

def annual_loss_prob(k: int, m: int, p_shard_fail: float = 0.01) -> float:
    n = k + m
    # Probability of losing MORE than m out of n shards.
    loss = 0.0
    for f in range(m + 1, n + 1):
        loss += comb(n, f) * (p_shard_fail ** f) * ((1 - p_shard_fail) ** (n - f))
    return loss

for k, m in [(1, 0), (1, 2), (10, 4), (17, 3)]:
    p = annual_loss_prob(k, m)
    nines = "inf" if p == 0 else f"{-log10(p):.1f}"
    print(f"k={k:>2} m={m:>1} -> annual loss prob = {p:.2e}   (~{nines} nines)")


k= 1 m=0 -> annual loss prob = 1.00e-02   (~2.0 nines)
k= 1 m=2 -> annual loss prob = 1.00e-06   (~6.0 nines)
k=10 m=4 -> annual loss prob = 1.86e-07   (~6.7 nines)
k=17 m=3 -> annual loss prob = 4.26e-05   (~4.4 nines)


Notice how `k=1, m=0` (no redundancy) is only 2 nines — terrible —
while `k=1, m=2` already buys us 6 nines. Real systems:

- spread the shards across **independent failure domains** (different
  racks, power zones, buildings) so failures aren't correlated,
- **repair quickly** — the window in which another failure can hurt
  you is shorter,
- and **scrub in the background** to catch silent bit-rot.

All three together push us to ~11 nines.


## 🗺️ High-level architecture

```
   [Client]
       │  HTTP + SigV4 / HMAC signature
       ▼
  ┌─────────────────────────────────────────┐
  │ API front-end (stateless, autoscaled)   │  parse, authN/authZ, rate-limit
  └──────────────┬──────────────────────────┘
         │       │                 │
         ▼       ▼                 ▼
     Auth/IAM  Metadata         Data plane
               (key → shard     (stores the actual bytes as
               locations)        erasure-coded shards on many
                                 storage nodes across racks/AZs)
                                         │
                                         ▼
                                Cross-region replication (async)
```

Two ideas carry most of the weight:

1. **Separate metadata from data.** Metadata is tiny but needs fast
   lookups and transactions — perfect for a sharded database.
   Data is huge but only needs sequential reads/writes — perfect for
   cheap commodity disks.
2. **Stateless front-ends**, stateful storage. Autoscale the pane we
   can; plan carefully for the pane we can't.


## ✅ What's next

- **Notebook 2** — a progressively-better toy S3 in Python: naive dict
  → versioned store → multipart upload.
- **Notebook 3** — deep dives: presigned URLs (with tamper tests),
  consistent hashing for shard placement, XOR-parity erasure coding,
  and a lifecycle policy engine.
